# 03 — Modelo de recompensa (Reward Model)

**Módulo:** `src/llm_rlhf/reward.py` → `RewardModel`, `RewardModelTrainer`

Tenemos un modelo SFT que genera respuestas con la *forma* de una instrucción. Algunas son buenas, la mayoría son mediocres. El trabajo del modelo de recompensa es asignar un *score escalar* a cualquier par `(prompt, response)` para que más tarde podamos usarlo como señal de entrenamiento.

## Setup (Colab o entorno nuevo)

Esta primera celda es **idempotente**: si `llm_rlhf` ya está instalado, no hace nada. En Colab, la primera vez clona el repositorio y lo instala en modo editable.

El repositorio apuntado es [emiliomunozai/LLM_RLHF](https://github.com/emiliomunozai/LLM_RLHF).

In [ ]:
# --- Colab / fresh-environment setup ---------------------------------
# No-op if llm_rlhf is already importable (e.g. local uv environment).
import os, sys, subprocess

REPO_URL = "https://github.com/emiliomunozai/LLM_RLHF.git"
REPO_DIR = "LLM_RLHF"

try:
    import llm_rlhf  # noqa: F401
except ImportError:
    if not os.path.exists(REPO_DIR):
        subprocess.run(["git", "clone", REPO_URL], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
    print("Installed llm_rlhf. If torch/transformers were upgraded, restart the runtime now.")

## ¿Por qué no calificar las respuestas a mano?

Una sola corrida de PPO puede muestrear millones de respuestas. Los humanos no pueden calificar tantas. Así que recogemos un lote *único* de valoraciones humanas, ajustamos un modelo que aproxima el juicio humano, y usamos *ese* modelo para puntuar los rollouts.

El modelo de recompensa es un proxy aprendido de la distribución de preferencias humanas. Su calidad pone un techo a la calidad de todo lo que venga después.

## Preferencias por pares, no calificaciones absolutas

Pedirle a un humano "califica esta respuesta del 1 al 10" produce ruido: distintas personas anclan sus puntajes de forma diferente, y la misma persona se desvía con el tiempo. Pedirle "¿cuál de estas dos es mejor?" es mucho más consistente.

Por eso los datos de entrenamiento son una lista de tripletas:

```
(prompt, y_preferred, y_dispreferred)
```

En este repositorio *simulamos* la etiqueta humana con una heurística (`simple_quality_score` en `reward.py`). En producción usarías anotadores reales — mira el dataset `Anthropic/hh-rlhf` como ejemplo.

## El modelo de Bradley–Terry

Si cada respuesta tiene una "calidad" latente $r(x, y)$, Bradley–Terry asume:

$$P(y_w \succ y_l \,|\, x) = \sigma\big(r(x, y_w) - r(x, y_l)\big)$$

Maximizar la log-verosimilitud de las preferencias observadas da la pérdida:

$$L = -\mathbb{E}\Big[\log \sigma\big(r(x, y_w) - r(x, y_l)\big)\Big]$$

El gradiente sobre esta pérdida empuja $r(y_w)$ hacia arriba y $r(y_l)$ hacia abajo cada vez que un humano eligió $y_w$ sobre $y_l$. Nada más — no hay escala absoluta, solo importa la *brecha* entre pares.

In [ ]:
import torch, torch.nn.functional as F

# The entire training objective in one line:
def bradley_terry_loss(r_preferred, r_dispreferred):
    return -F.logsigmoid(r_preferred - r_dispreferred).mean()

# Sanity check: when preferred wins by 1 logit, loss is small; when it loses, loss is large.
for gap in [-2.0, 0.0, 2.0]:
    loss = bradley_terry_loss(torch.tensor(gap), torch.tensor(0.0)).item()
    print(f'reward_gap={gap:+.1f} -> loss={loss:.3f}')

### La pérdida, visualizada

La fórmula es compacta, pero la intuición vale más cuando se ve. La siguiente celda grafica la pérdida en función del *gap* $r(y_w) - r(y_l)$:

- Cuando el gap es muy negativo (el modelo prefiere lo malo): pérdida alta → gradiente fuerte.
- Cuando el gap es cero (modelo indeciso): pérdida = $\log 2 pprox 0.69$.
- Cuando el gap es muy positivo (modelo correcto y seguro): pérdida → 0.

Esta forma es exactamente la del log-loss binario — Bradley–Terry *es* regresión logística entre dos respuestas.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Plot the Bradley-Terry loss as a function of the reward gap.
gaps = np.linspace(-5, 5, 200)
losses = [bradley_terry_loss(torch.tensor(float(g)), torch.tensor(0.0)).item() for g in gaps]

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(gaps, losses, linewidth=2)
ax.axvline(0, color='gray', linestyle='--', alpha=0.6, label='gap = 0 (no preference learned)')
ax.set_xlabel('reward gap  r(y_preferred) − r(y_dispreferred)')
ax.set_ylabel('Bradley-Terry loss')
ax.set_title('Loss vs. reward gap — the gradient pushes the gap to the right')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Arquitectura: un LM con una cabeza escalar

El modelo de recompensa **reutiliza el backbone transformer** del policy. Borramos la cabeza de lenguaje (LM head) y atornillamos una única capa lineal que proyecta el último estado oculto a un escalar.

¿El estado oculto de qué token? Usamos el *último token no-padding*. Para esa posición, la atención ya vio todo el `prompt + response`, así que la representación es un resumen de la interacción completa.

Mira `RewardModel.forward` en `reward.py` — son unas 15 líneas.

In [ ]:
from llm_rlhf import PretrainedLLM
from llm_rlhf.reward import RewardModel

llm = PretrainedLLM()
rm = RewardModel(llm)
print(rm.reward_head)  # the only new parameters: Linear(hidden, 1)

## Visualizando la heurística sobre nuestro caso canónico

`simple_quality_score` es nuestro "anotador humano sintético". No es bueno — premia preguntas retóricas, palabras "conversacionales" y longitudes razonables. Pero produce una señal *consistente*, que es lo único que el modelo de recompensa necesita para aprender preferencias.

Para hilar con las etapas anteriores, vamos a puntuar **respuestas plausibles al `CANONICAL_PROMPT`** que cubren el espectro entre el modelo base y un modelo SFT bien entrenado. Mira el ranking que produce — es justo el orden que el modelo de recompensa intentará reproducir.

In [ ]:
from llm_rlhf import CANONICAL_PROMPT
from llm_rlhf.reward import simple_quality_score

# Four plausible completions for the canonical prompt — spanning the spectrum
# we'd actually see across stages of this pipeline:
#   base       : continues like an article, doesn't answer
#   sft-terse  : answers but with no warmth
#   sft-engaging: answers and uses analogies + 'imagine'
#   sft-broken : technically answers but is incoherent
prompt = CANONICAL_PROMPT
candidate_responses = [
    # base — never actually answers; reads like the start of a wiki page
    "Quantum computing is the same thing as the computer used in the Internet. But how does a quantum computer work? First, you have a quantum computer.",
    # SFT, terse — correct form, no warmth (the heuristic dislikes this)
    "Quantum computers use qubits instead of bits.",
    # SFT, engaging — what we'd hope RLHF gets us
    "Imagine a normal computer is like a light switch that's either on or off. A quantum computer is more like a spinning coin — it can be a little of both at once. Think of it as having many switches turned on at the same time, which lets it try many answers at once! Wow, right?",
    # SFT, broken — answers grammatically but no actual content
    "Quantum computing computer quantum thing okay bye",
]

scores = [(r, simple_quality_score(r)) for r in candidate_responses]
scores.sort(key=lambda x: -x[1])

print(f'Prompt: {prompt}')
print('-' * 60)
for r, s in scores:
    print(f'score={s:+.1f}  |  {r[:90]}{"..." if len(r) > 90 else ""}')

In [ ]:
import matplotlib.pyplot as plt

texts  = [r for r, _ in scores]
values = [s for _, s in scores]

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(range(len(texts)), values[::-1], color='#4a90d9')
ax.set_yticks(range(len(texts)))
ax.set_yticklabels([t[:55] + ('...' if len(t) > 55 else '') for t in texts[::-1]], fontsize=8)
ax.set_xlabel('simple_quality_score')
ax.set_title('Heuristic ranking — this is the proxy for "human" preferences')
plt.tight_layout()
plt.show()

## Cómo se ven los datos de entrenamiento

El modelo de recompensa **no se entrena con scores absolutos** — los scores arriba son solo una intermediación para construir *pares*. La unidad de entrenamiento del RM es:

```
(prompt, y_w, y_l)   # y_w = preferida, y_l = no preferida
```

`RewardModelTrainer.create_preference_dataset` toma muchas respuestas por prompt, las puntúa con el scorer, y emite todos los pares ordenables (mayor score gana). De 4 candidatos salen hasta $\binom{4}{2} = 6$ pares — la siguiente celda lo muestra explícitamente para nuestro `CANONICAL_PROMPT`.

In [ ]:
# Construct the actual preference pairs the RM would train on.
# Same logic as RewardModelTrainer.create_preference_dataset, condensed inline.
pairs = []
for i in range(len(candidate_responses)):
    for j in range(i + 1, len(candidate_responses)):
        si = simple_quality_score(candidate_responses[i])
        sj = simple_quality_score(candidate_responses[j])
        if si > sj:
            pairs.append((prompt, candidate_responses[i], candidate_responses[j]))
        elif sj > si:
            pairs.append((prompt, candidate_responses[j], candidate_responses[i]))

print(f"Built {len(pairs)} preference pairs from {len(candidate_responses)} responses.\n")
for k, (p, w, l) in enumerate(pairs[:3]):  # show the first few
    print(f"--- pair {k+1} ---")
    print(f"PROMPT: {p}")
    print(f"WIN  : {w[:80]}{'...' if len(w) > 80 else ''}")
    print(f"LOSE : {l[:80]}{'...' if len(l) > 80 else ''}")
    print()

## Lo que el modelo de recompensa *no* te dice

- **Calidad absoluta.** La salida no tiene unidades. Una recompensa de 3.7 no es "mejor" que 2.1 en ningún sentido humano interpretable — solo importa la *brecha* entre respuestas al mismo prompt.
- **Calibración con humanos.** Aproxima las preferencias de los anotadores, incluyendo sus sesgos. Etiquetas basura entran → recompensas basura salen.
- **Puntuaciones fuera de distribución.** Respuestas que no se parecen en nada a los datos de entrenamiento pueden obtener puntuaciones arbitrarias. Esta es la fuente del *reward hacking* en PPO — el policy encuentra salidas absurdas que puntúan alto.

## Ejercicio: ¿estás de acuerdo con la heurística?

La forma más rápida de aprender qué limita un modelo de recompensa es construir uno mentalmente y contrastarlo con el del repositorio:

1. Escribe 3–4 respuestas candidatas a un prompt de tu elección.
2. Puntúalas con `simple_quality_score`.
3. Ordénalas tú mismo a mano. ¿Dónde estás en desacuerdo con la heurística?

Cada desacuerdo revela *exactamente* un modo de fallo que el modelo de recompensa va a heredar (y que PPO va a explotar como reward hacking).

In [ ]:
# Exercise: do *you* agree with the heuristic?
# 1. Write 3-4 candidate responses to a prompt of your choice.
# 2. Score them with simple_quality_score.
# 3. Rank them yourself by hand. Where do you disagree with the heuristic?
# Disagreements reveal *exactly* the failure modes the reward model will inherit.

your_prompt = "How does photosynthesis work?"
your_responses = [
    "Plants eat sunlight.",
    "Plants use chlorophyll to convert CO2 and water into glucose, releasing oxygen.",
    "Wow, photosynthesis is amazing! Imagine if you could eat sunshine.",
    # add more...
]

for r in your_responses:
    print(f'{simple_quality_score(r):+5.1f}  |  {r}')

## Próximos pasos — encadenando la pipeline

Tres caminos usan este modelo de recompensa (cuando lo entrenes, se guarda en `checkpoints/reward/`):

- **`04_ppo.ipynb`** — úsalo como señal de recompensa en RL. PPO carga el adaptador SFT *y* este RM.
- **`05_dpo.ipynb`** — *evítalo entrenar por completo* re-parametrizando la recompensa en términos del propio policy. Solo necesita el SFT y los *mismos pares de preferencias* que viste arriba.
- **`06_grpo.ipynb`** — úsalo para puntuar grupos de respuestas y calcular ventajas relativas. También carga SFT + RM.

> **Realidad observada con OPT-350M y `simple_quality_score`:** tras entrenar 5 epochs sobre 82 pares, el RM converge a accuracy ≈ 66 % (no a 100 %) y reward_gap ≈ +0.6. Es un modelo **decente pero no perfecto** — y eso se traslada aguas abajo: PPO/GRPO heredan ese 34 % de ruido como su techo. Reduces el ruido aumentando los pares (regenera con `num_prompts=100` o más), no afinando hiperparámetros del RM.
>
> **Curiosidad reveladora:** al puntuar las salidas Base ("What's quantum computing?", una pregunta) y SFT (un párrafo largo), nuestro RM dio +1.47 a Base y +1.31 a SFT — *castiga* a la respuesta más informativa. Esto refleja el sesgo de `simple_quality_score` por los signos de interrogación. Es exactamente el tipo de fallo que vas a ver explotado en PPO/GRPO (reward hacking) — el policy aprenderá a producir más preguntas porque eso sube la recompensa, aunque sea pedagógicamente peor. *Garbage in, garbage out* en miniatura.